In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import anndata as ad

In [2]:

per_dataset_files = {
    "1": "/rds/general/user/ztb25/home/PBMC_datasets/1/PBMC1_clustering.h5ad",
    "4": "/rds/general/user/ztb25/home/PBMC_datasets/4/PBMC4_clustering.h5ad",
    "5": "/rds/general/user/ztb25/home/PBMC_datasets/5/PBMC5_clustering.h5ad",
    "6": "/rds/general/user/ztb25/home/PBMC_datasets/6/PBMC6_clustering.h5ad",
}

donor_tables = []

for dataset_id, file_path in per_dataset_files.items():
    # backed="r" reads obs only, the count matrix stays on disk (fast, low memory)
    adata_backed = ad.read_h5ad(file_path, backed="r")
    obs = adata_backed.obs.copy()
    adata_backed.file.close()

    # dataset 6 (GSE275067) is control-only and carries no diagnosis column
    if "diagnosis" not in obs.columns:
        obs["diagnosis"] = "CTRL"
    # ethnicity is only reported by dataset 6
    if "race" not in obs.columns:
        obs["race"] = pd.NA

    # sanity check: 'sample' must be donor-level, i.e. one age/gender/diagnosis per sample.
    # if this trips, 'sample' is a library ID not a donor ID and the counts below are wrong.
    values_per_sample = obs.groupby("sample", observed=True)[["age", "gender", "diagnosis"]].nunique()
    inconsistent_samples = values_per_sample[values_per_sample.gt(1).any(axis=1)]
    if len(inconsistent_samples) > 0:
        print(f"WARNING dataset {dataset_id}: {len(inconsistent_samples)} samples have "
              f"conflicting metadata:\n{inconsistent_samples}")

    # collapse cells -> one row per donor. Doing this BEFORE any statistics is the whole point:
    # cell-level means are weighted by how many cells each donor contributed, not by donor.
    donor_level = (
        obs[["sample", "diagnosis", "age", "gender", "race"]]
        .drop_duplicates(subset="sample")
        .copy()
    )
    donor_level["dataset"] = dataset_id
    donor_level["n_cells"] = obs["sample"].map(obs["sample"].value_counts()).groupby(obs["sample"]).first().reindex(donor_level["sample"]).values
    donor_tables.append(donor_level)

donors = pd.concat(donor_tables, ignore_index=True)

# --- harmonise coding across datasets -------------------------------------
# ds4 uses male/female, ds5 uses M/F, ds6 uses Male/Female. Same map you already
# use in differential_expression.ipynb, kept identical so the tables agree.
gender_map = {"M": "male", "F": "female",
              "Male": "male", "Female": "female",
              "male": "male", "female": "female"}
donors["gender_raw"] = donors["gender"].astype(str).str.strip()
donors["gender"] = donors["gender_raw"].map(gender_map)

unmapped_gender = donors.loc[donors["gender"].isna(), "gender_raw"].unique()
if len(unmapped_gender) > 0:
    print("WARNING unmapped gender labels:", unmapped_gender)

donors["age"] = pd.to_numeric(donors["age"], errors="coerce")
print(f"donors with missing age: {donors['age'].isna().sum()} / {len(donors)}")

donors.head()


donors with missing age: 0 / 103


,sample,diagnosis,age,gender,race,dataset,n_cells,gender_raw
0,1,CTRL,67.0,female,NaN,1,9455,F
1,2,CTRL,73.0,female,NaN,1,6997,F
2,3,PD,66.0,female,NaN,1,13240,F
3,4,PD,66.0,female,NaN,1,8117,F
4,5,PD,78.0,female,NaN,1,9090,F


In [3]:
# one row per dataset, donor-level age stats (donors with missing age excluded from the stats
# but counted in age_missing, so n_donors and the stats don't silently disagree)
age_by_dataset = (
    donors.groupby("dataset", observed=True)
          .agg(n_donors    = ("age", "size"),
               n_with_age  = ("age", "count"),      # count() skips NaN
               age_missing = ("age", lambda ages: ages.isna().sum()),
               age_mean    = ("age", "mean"),
               age_sd      = ("age", "std"),        # ddof=1 by default
               age_median  = ("age", "median"),
               age_min     = ("age", "min"),
               age_max     = ("age", "max"))
          .round(1)
)

# convenience column for pasting straight into the dissertation table
age_by_dataset["age_range"] = (
    age_by_dataset["age_min"].astype("Int64").astype(str) + "–" +
    age_by_dataset["age_max"].astype("Int64").astype(str)
)

display(age_by_dataset)


,n_donors,n_with_age,age_missing,age_mean,age_sd,age_median,age_min,age_max,age_range
dataset,,,,,,,,,
1,6,6,0,71.0,5.4,70.0,66.0,78.0,66–78
4,50,50,0,72.7,9.4,73.0,47.0,89.0,47–89
5,24,24,0,67.5,6.3,67.0,52.0,82.0,52–82
6,23,23,0,71.2,8.9,70.0,55.0,88.0,55–88


In [4]:
# counts
gender_by_dataset = (
    donors.groupby(["dataset", "gender"], observed=True)
          .size()
          .unstack(fill_value=0)
)
gender_by_dataset["total"]      = gender_by_dataset.sum(axis=1)
gender_by_dataset["pct_female"] = (100 * gender_by_dataset["female"] / gender_by_dataset["total"]).round(1)

display(gender_by_dataset)


gender,female,male,total,pct_female
dataset,,,,
1,6,0,6,100.0
4,24,26,50,48.0
5,12,12,24,50.0
6,14,9,23,60.9


In [5]:
# whole-cohort age stats: same donor-level table, no grouping
integrated_ages = donors["age"].dropna()

print(f"n donors with age: {len(integrated_ages)} / {len(donors)} "
      f"({donors['age'].isna().sum()} missing)")
print(f"mean (SD):    {integrated_ages.mean():.1f} ({integrated_ages.std(ddof=1):.1f})")
print(f"median [IQR]: {integrated_ages.median():.1f} "
      f"[{integrated_ages.quantile(.25):.1f}–{integrated_ages.quantile(.75):.1f}]")
print(f"range:        {integrated_ages.min():.0f}–{integrated_ages.max():.0f}")


n donors with age: 103 / 103 (0 missing)
mean (SD):    71.0 (8.6)
median [IQR]: 70.0 [65.0–77.0]
range:        47–89
